In [1]:
import pandas as pd 
import torch.nn as nn
import torch 
import torch.optim as optim
import numpy as np 
from torch.utils.data import DataLoader, TensorDataset

In [2]:
df = pd.read_csv(rf"E:\AIML_apna_college\07_Pytorch_DL\datasets\DateFruit_Dataset.csv")

In [3]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler ,LabelEncoder

In [6]:
X = df.drop("Class",axis=1)
y = df["Class"]

In [7]:
le = LabelEncoder()
y = le.fit_transform(y)


In [8]:
X_train , X_test , y_train ,y_test = train_test_split(
    X, y , test_size=.2 ,random_state=42
)


In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ANN

## Why Use dtype=torch.long for Labels in Classification?

### 1. Cross-Entropy Loss Requirement
PyTorch's nn.CrossEntropyLoss expects:
- Predictions: Float tensor with raw logits (unnormalized scores)
- Labels: Long integer tensor with class indices (0, 1, 2, ...)

Cross-entropy internally applies log_softmax to predictions and uses label indices to select correct class probabilities. Labels act as array indices, requiring integer type.

### 2. How Cross-Entropy Uses Labels Internally
```python
# Simplified internal process
log_probs = log(softmax(predictions))
loss = -log_probs[label]  # Label used as INDEX
```
Example with 3 classes:
```python
predictions = [[2.3, 0.5, 1.2]]  # Raw logits
label = [1]  # Class index 1 (must be integer!)
# CrossEntropyLoss selects prediction at index 1
```

### 3. Integer Indexing in Classification
In multi-class problems, each label represents a discrete class. Labels are used as indices to access specific neurons in the output layer. torch.long (64-bit integer) ensures precise indexing without floating-point errors.

### 4. Relationship to ANN Architecture
Output layer has one neuron per class. Loss function needs to know which output neuron corresponds to the true class. If true class is 1, it maximizes score of neuron at index 1. This requires integer indexing.

### 5. Common Error
```python
# WRONG - causes error
y_train = torch.tensor(y_train, dtype=torch.float32)
# Error: Expected scalar type Long but got Float

# CORRECT
y_train = torch.tensor(y_train, dtype=torch.long)
```

### Key Takeaway
- Features (X): torch.float32 (continuous values)
- Labels (y): torch.long (discrete class indices)
- CrossEntropyLoss uses labels as array indices
- DO NOT one-hot encode labels with CrossEntropyLoss

In [10]:
X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(X_test_scaled,dtype=torch.long)

In [19]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [20]:
train_loader = DataLoader(train_dataset,shuffle=True ,batch_size=32)
test_loader = DataLoader(test_dataset,batch_size=32)

Model 

In [21]:
# ANN model for classification 
class ANN_C(nn.Module):
    def __init__(self,):
        super(ANN_C,self).__init__()
        # first layer 
        self.model = nn.Sequential(
            nn.Linear(X.shape[1],64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64, 7)
        )
    def forward(self,x):
        return self.model(x)


In [23]:
model = ANN_C()

creteria = nn.CrossEntropyLoss(
)
optimizer = optim.Adam(model.parameters())



In [24]:
epochs = 100 
epoch =1
for (epoch) in  range(epochs+1):
    model.train()
    running_loss = 0.0

    for xb ,yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss =creteria(outputs,yb)
        loss.backward()
        optimizer.step()

        running_loss +=loss
    training_loss = running_loss / len(train_loader)
    print(f"epoch {epoch}/{epochs} , loss = {training_loss}")


epoch 0/100 , loss = 1.7132670879364014
epoch 1/100 , loss = 1.1333307027816772
epoch 2/100 , loss = 0.7274057269096375
epoch 3/100 , loss = 0.5397793054580688
epoch 4/100 , loss = 0.44602465629577637
epoch 5/100 , loss = 0.3895605802536011
epoch 6/100 , loss = 0.3483245372772217
epoch 7/100 , loss = 0.32744327187538147
epoch 8/100 , loss = 0.28816595673561096
epoch 9/100 , loss = 0.26777875423431396
epoch 10/100 , loss = 0.24927914142608643
epoch 11/100 , loss = 0.23677830398082733
epoch 12/100 , loss = 0.21561601758003235
epoch 13/100 , loss = 0.21886523067951202
epoch 14/100 , loss = 0.203102707862854
epoch 15/100 , loss = 0.1961957961320877
epoch 16/100 , loss = 0.19414272904396057
epoch 17/100 , loss = 0.1739872246980667
epoch 18/100 , loss = 0.17011559009552002
epoch 19/100 , loss = 0.16477134823799133
epoch 20/100 , loss = 0.17014597356319427
epoch 21/100 , loss = 0.15971225500106812
epoch 22/100 , loss = 0.156514510512352
epoch 23/100 , loss = 0.14816798269748688
epoch 24/100 ,

In [26]:
# fix test labels and rebuild test loader
y_test_tensor = torch.tensor(y_test, dtype=torch.long)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32)

# evaluation
model.eval()
total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted_index = torch.max(outputs, 1)
        correct += (predicted_index == yb).sum().item()
        total += yb.size(0)

print("total value :", total)
print("correct value :", correct)
print("accuracy :", 100 * correct / total)



        


total value : 180
correct value : 174
accuracy : 96.66666666666667
